# 📊 YouTube Comment Sentiment Analyzer

Notebook ini menganalisis sentimen komentar publik pada video YouTube secara otomatis.  
Pipeline: **Data Fetch → Preprocessing → Sentiment Scoring → Visualization**

> ⚠️ **Catatan:** Analisis sentimen menggunakan [TextBlob](https://textblob.readthedocs.io/) berdasarkan skor polaritas (positif/netral/negatif).

---

## 1 · Setup & Konfigurasi
Instal dependensi (jika belum) dan impor semua pustaka yang dibutuhkan.

In [ ]:
# ── Install Dependencies ──────────────────────────────────────────────────────
# Uncomment baris di bawah jika library belum terinstal:
# !pip install google-api-python-client pandas matplotlib seaborn textblob transformers torch

# ── Imports ───────────────────────────────────────────────────────────────────
import re
import warnings
import os
from getpass import getpass

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
from googleapiclient.discovery import build
from textblob import TextBlob
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# ── Styling ───────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
})

# ── Konfigurasi ───────────────────────────────────────────────────────────────
load_dotenv()
API_KEY = os.getenv('API_KEY')
if not API_KEY:
    API_KEY = getpass('🔑 Masukkan YouTube Data API v3 Key: ')
else:
    print('✅ API Key dimuat dari file .env')
VIDEO_INPUT = input('🎬 Masukkan URL atau Video ID YouTube: ')

# Ekstrak Video ID dari URL (jika user memasukkan full URL)
def extract_video_id(url_or_id: str) -> str:
    """Ekstrak video ID dari berbagai format URL YouTube."""
    patterns = [
        r'(?:v=|/v/|youtu\.be/|/live/)([a-zA-Z0-9_-]{11})',  # Standard, shortened, & live URLs
        r'(?:embed/)([a-zA-Z0-9_-]{11})',               # Embed URLs
        r'^([a-zA-Z0-9_-]{11})$',                        # Raw video ID
    ]
    for pattern in patterns:
        match = re.search(pattern, url_or_id)
        if match:
            return match.group(1)
    raise ValueError(f'❌ Tidak dapat mengekstrak Video ID dari: {url_or_id}')

VIDEO_ID = extract_video_id(VIDEO_INPUT)
print(f'\n✅ Video ID: {VIDEO_ID}')

## 2 · Data Acquisition Layer
Mengambil komentar teratas (*top-level comments*) dari video menggunakan **YouTube Data API v3** dengan dukungan paginasi.

In [ ]:
def fetch_comments(api_key: str, video_id: str, max_comments: int = None) -> pd.DataFrame:
    """
    Mengambil komentar top-level dari video YouTube.

    Parameters
    ----------
    api_key : str
        YouTube Data API v3 key.
    video_id : str
        ID video YouTube (11 karakter).
    max_comments : int, default 500
        Jumlah maksimum komentar yang diambil.

    Returns
    -------
    pd.DataFrame
        DataFrame berisi kolom: author, comment, published_at, like_count.
    """
    youtube = build('youtube', 'v3', developerKey=api_key)
    comments = []
    next_page_token = None

    print('📥 Mengambil komentar', end='')
    while True:
        response = youtube.commentThreads().list(
            part='snippet',
            videoId=video_id,
            textFormat='plainText',
            maxResults=100,
            pageToken=next_page_token
        ).execute()

        for item in response['items']:
            snippet = item['snippet']['topLevelComment']['snippet']
            comments.append({
                'author': snippet['authorDisplayName'],
                'comment': snippet['textDisplay'],
                'published_at': snippet['publishedAt'],
                'like_count': snippet['likeCount'],
            })

        print('.', end='', flush=True)
        next_page_token = response.get('nextPageToken')

        if not next_page_token or (max_comments is not None and len(comments) >= max_comments):
            break

    print(f' Selesai!')
    df = pd.DataFrame(comments[:max_comments] if max_comments is not None else comments)
    df['published_at'] = pd.to_datetime(df['published_at'])
    return df


# ── Eksekusi ──────────────────────────────────────────────────────────────────
df = fetch_comments(API_KEY, VIDEO_ID, max_comments=None)
print(f'\n📋 Total komentar diambil: {len(df):,}')
df.head()

## 3 · Preprocessing & Analisis Sentimen
Membersihkan teks komentar dari *noise* (URL, mention, HTML, karakter aneh) dan mengklasifikasikan sentimen menggunakan **TextBlob** berdasarkan skor polaritas.
- **Positif** → polaritas > 0
- **Netral** → polaritas = 0
- **Negatif** → polaritas < 0


In [ ]:
# ── Text Cleaning ─────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    Membersihkan teks komentar dari noise.

    Langkah-langkah:
    1. Hapus URL (http/https/www)
    2. Hapus mention (@username)
    3. Hapus tag HTML
    4. Hapus karakter non-alfanumerik (kecuali spasi & tanda baca umum)
    5. Normalkan spasi
    """
    text = re.sub(r'http\S+|www\.\S+', '', text)       # Hapus URL
    text = re.sub(r'@\w+', '', text)                    # Hapus @mention
    text = re.sub(r'<.*?>', '', text)                    # Hapus HTML tags
    text = re.sub(r'[^\w\s.,!?\'-]', '', text)          # Hapus karakter aneh
    text = re.sub(r'\s+', ' ', text).strip()             # Normalkan spasi
    return text


# ── Sentiment Classification ──────────────────────────────────────────────────
def get_sentiment_label(polarity: float) -> str:
    """Klasifikasi skor polaritas ke label sentimen."""
    if polarity > 0:
        return 'Positif'
    elif polarity < 0:
        return 'Negatif'
    else:
        return 'Netral'


# ── Pipeline ──────────────────────────────────────────────────────────────────
df['clean_comment'] = df['comment'].apply(clean_text)
df['polarity'] = df['clean_comment'].apply(lambda t: TextBlob(t).sentiment.polarity)
df['subjectivity'] = df['clean_comment'].apply(lambda t: TextBlob(t).sentiment.subjectivity)
df['sentiment'] = df['polarity'].apply(get_sentiment_label)

# ── Hasil ─────────────────────────────────────────────────────────────────────
print('🔍 Distribusi Sentimen:')
print(df['sentiment'].value_counts())
print(f'\n📊 Rata-rata Polarity: {df["polarity"].mean():.4f}')
print(f'📊 Rata-rata Subjectivity: {df["subjectivity"].mean():.4f}')

df[['author', 'clean_comment', 'polarity', 'sentiment']].head(10)

## 4 · Analytics & Visualisasi Data
Menyajikan hasil analisis sentimen dalam bentuk grafik visual:
1. **Pie Chart** — Distribusi persentase sentimen secara makro
2. **Bar Chart** — Perbandingan frekuensi riil jumlah komentar per kategori

In [ ]:
# ── Warna & Konfigurasi ──────────────────────────────────────────────────────
SENTIMENT_ORDER = ['Positif', 'Netral', 'Negatif']
COLORS = {
    'Positif': '#2ecc71',   # Hijau segar
    'Netral':  '#95a5a6',   # Abu-abu netral
    'Negatif': '#e74c3c',   # Merah tegas
}

sentiment_counts = df['sentiment'].value_counts()
# Pastikan urutan konsisten
sentiment_counts = sentiment_counts.reindex(SENTIMENT_ORDER).fillna(0).astype(int)
color_list = [COLORS[s] for s in sentiment_counts.index]

# ═════════════════════════════════════════════════════════════════════════════
# FIGURE: Dual Chart — Pie + Bar
# ═════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Analisis Sentimen Komentar YouTube',
             fontsize=20, fontweight='bold', y=1.02)

# ── 1. Pie Chart ─────────────────────────────────────────────────────────────
ax1 = axes[0]
explode = [0.05 if v == sentiment_counts.max() else 0 for v in sentiment_counts.values]

wedges, texts, autotexts = ax1.pie(
    sentiment_counts.values,
    labels=sentiment_counts.index,
    colors=color_list,
    autopct='%1.1f%%',
    startangle=140,
    explode=explode,
    shadow=True,
    textprops={'fontsize': 13},
    pctdistance=0.75,
)

# Styling autopct text
for autotext in autotexts:
    autotext.set_fontweight('bold')
    autotext.set_color('white')

# Donut style — tambahkan lingkaran putih di tengah
centre_circle = plt.Circle((0, 0), 0.50, fc='white')
ax1.add_artist(centre_circle)
ax1.set_title('Distribusi Sentimen (%)', pad=20)

# ── 2. Bar Chart ─────────────────────────────────────────────────────────────
ax2 = axes[1]
bars = ax2.bar(
    sentiment_counts.index,
    sentiment_counts.values,
    color=color_list,
    edgecolor='white',
    linewidth=1.5,
    width=0.6,
)

# Label di atas setiap bar
for bar in bars:
    height = bar.get_height()
    ax2.text(
        bar.get_x() + bar.get_width() / 2.,
        height + (sentiment_counts.max() * 0.02),
        f'{int(height):,}',
        ha='center', va='bottom',
        fontweight='bold', fontsize=14,
    )

ax2.set_title('Jumlah Komentar per Kategori', pad=20)
ax2.set_xlabel('Sentimen', fontsize=13)
ax2.set_ylabel('Jumlah Komentar', fontsize=13)
ax2.set_ylim(0, sentiment_counts.max() * 1.15)  # Ruang untuk label
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('sentiment_analysis_result.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()

print('\n💾 Grafik disimpan ke: sentiment_analysis_result.png')

## 5 · Ringkasan & Sampel Output
Statistik agregat dan contoh komentar dari setiap kategori sentimen.

In [ ]:
# ── Statistik Agregat ────────────────────────────────────────────────────────
total = len(df)
print('=' * 60)
print('           📊 RINGKASAN ANALISIS SENTIMEN')
print('=' * 60)
print(f'  Total Komentar Dianalisis  : {total:,}')
print(f'  Rata-rata Polarity Score   : {df["polarity"].mean():+.4f}')
print(f'  Rata-rata Subjectivity     : {df["subjectivity"].mean():.4f}')
print('-' * 60)

for sentiment in ['Positif', 'Netral', 'Negatif']:
    count = len(df[df['sentiment'] == sentiment])
    pct = (count / total) * 100 if total > 0 else 0
    emoji = {'Positif': '🟢', 'Netral': '⚪', 'Negatif': '🔴'}[sentiment]
    print(f'  {emoji} {sentiment:<10} : {count:>5,} komentar  ({pct:5.1f}%)')

print('=' * 60)

# ── Sampel Komentar per Kategori ─────────────────────────────────────────────
print('\n\n📝 Contoh Komentar per Kategori:\n')

for sentiment in ['Positif', 'Netral', 'Negatif']:
    subset = df[df['sentiment'] == sentiment].nlargest(3, 'like_count')
    emoji = {'Positif': '🟢', 'Netral': '⚪', 'Negatif': '🔴'}[sentiment]
    print(f'── {emoji} {sentiment.upper()} (Top 3 by likes) ─────────────────')
    if subset.empty:
        print('   Tidak ada komentar dalam kategori ini.')
    else:
        for _, row in subset.iterrows():
            comment_preview = row['clean_comment'][:100]
            if len(row['clean_comment']) > 100:
                comment_preview += '...'
            print(f'   👤 {row["author"]}')
            print(f'   💬 "{comment_preview}"')
            print(f'   👍 {row["like_count"]:,} likes  |  Polarity: {row["polarity"]:+.3f}')
            print()

# ── Export ke CSV (Opsional) ──────────────────────────────────────────────────
output_file = f'sentiment_results_{VIDEO_ID}.csv'
df.to_csv(output_file, index=False)
print(f'\n💾 Hasil lengkap diekspor ke: {output_file}')